# 9. Is the model looking at the disease, or at the camera?

In DeepDRiD, the larger photos are far more often referable than the smaller ones. Image size may point to a different camera or clinic, although the dataset doesn't confirm that.

A model could therefore score well partly by recognising the type of photo. That wouldn't help in a Kenyan village, so we check for it here.

It compares every model it finds in `results`, so run it again after adding the neural networks from notebook 8.

In [ ]:
%pip install -q numpy pandas matplotlib scikit-learn

In [ ]:
from pathlib import Path

# Finds the project folder wherever this notebook is opened from.
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data" / "raw" / "DeepDRiD").exists()), None)
if ROOT is None:
    raise FileNotFoundError("Could not find the AfriEyeSamsung project folder. Open this notebook from inside it.")
PROCESSED, RESULTS, MODELS = ROOT / "data" / "processed", ROOT / "results", ROOT / "models"
print("Project folder:", ROOT)

In [ ]:
import json

import numpy as np
import pandas as pd
from sklearn.metrics import confusion_matrix, roc_auc_score

metrics = {f.name.replace("_metrics.json", ""): json.loads(f.read_text()) for f in sorted(RESULTS.glob("*_metrics.json"))}
predictions = {slug: pd.read_csv(RESULTS / f"{slug}_predictions.csv") for slug in metrics}
labels = pd.read_csv(PROCESSED / "clean_labels.csv")
labels["image_size"] = labels.width.astype(int).astype(str) + " x " + labels.height.astype(int).astype(str)
for slug in predictions:
    predictions[slug] = predictions[slug].merge(labels[["image_id", "image_size"]], on="image_id", validate="one_to_one")
print("Models found:", ", ".join(m["model"] for m in metrics.values()))
print("Not run yet:", ", ".join(s for s in ["xgboost", "mobilenet_v3", "efficientnet_b0", "convnext_tiny"] if s not in metrics) or "none")

In [ ]:
import matplotlib.pyplot as plt

BLUE, TEAL, ORANGE, PURPLE, GREY = "#1F4E9E", "#00948F", "#C4581F", "#7A3E9D", "#8A94A6"
INK, MUTED, GRID = "#1B2433", "#566173", "#E3E7F0"
plt.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": 110, "font.size": 10, "axes.titlesize": 12, "axes.titleweight": "bold",
    "axes.edgecolor": "#AEB6C5", "axes.labelcolor": MUTED, "axes.titlecolor": INK, "xtick.color": MUTED, "ytick.color": MUTED,
    "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True, "grid.color": GRID, "axes.axisbelow": True,
    "legend.frameon": False,
})

## Check 1: how strong is the difference?

If image size alone gives a high AUC, then a model can look good without seeing any disease.

In [ ]:
scored = labels[labels.split.isin(["val", "test"])]
common = labels.image_size.value_counts().head(2).index.tolist()
summary = labels[labels.image_size.isin(common)].groupby(["split", "image_size"]).referable.agg(Photos="count", Referable="mean").reset_index()
summary["Referable"] = (summary.Referable * 100).round(1)
print(summary.to_string(index=False))
print("Photos with other sizes:", int((~labels.image_size.isin(common)).sum()))

known = labels[labels.image_size.isin(common)]
larger = (known.image_size == sorted(common)[-1]).astype(int)
size_auc = {s: roc_auc_score(known.referable[known.split == s], larger[known.split == s]) for s in ("train", "val", "test")}
print("AUC from image size alone, with no photo content:", {k: round(v, 3) for k, v in size_auc.items()})

## Check 2: AUC inside each image size

Comparing photos of the same size removes the shortcut. A big fall means part of the overall score came from photo type, not disease.

The range is a 95% interval that resamples whole patients. With few patients, these ranges are wide.

In [ ]:
def patient_ci(y, score, patients, repeats=500, seed=0):
    rng = np.random.default_rng(seed)
    y, score, patients = np.asarray(y), np.asarray(score), np.asarray(patients)
    ids = np.unique(patients)
    where = {p: np.where(patients == p)[0] for p in ids}
    results = []
    for _ in range(repeats):
        take = np.concatenate([where[p] for p in rng.choice(ids, len(ids))])
        if 0 < y[take].sum() < len(take):
            results.append(roc_auc_score(y[take], score[take]))
    return float(np.percentile(results, 2.5)), float(np.percentile(results, 97.5))


rows = []
for slug, p in predictions.items():
    for split in ("val", "test"):
        part = p[p.split == split]
        overall = roc_auc_score(part.referable, part.score)
        for size, g in part[part.image_size.isin(common)].groupby("image_size"):
            low, high = patient_ci(g.referable, g.score, g.patient_id)
            rows.append({"slug": slug, "Model": metrics[slug]["model"], "Split": split, "Image size": size, "Photos": len(g), "% referable": g.referable.mean() * 100,
                         "Overall AUC": overall, "AUC in this size": roc_auc_score(g.referable, g.score), "Low": low, "High": high})
within = pd.DataFrame(rows)
order = within[within.Split == "val"].groupby("Model")["Overall AUC"].first().sort_values(ascending=False).index.tolist()
view = within[within.Split == "test"].drop(columns=["slug", "Split"]).set_index("Model").loc[order].reset_index()
print(view.to_string(index=False, formatters={"% referable": "{:.0f}".format, "Overall AUC": "{:.3f}".format, "AUC in this size": "{:.3f}".format, "Low": "{:.2f}".format, "High": "{:.2f}".format}))

In [ ]:
sizes = sorted(common)
fig, ax = plt.subplots(figsize=(8.6, max(3, .85 * len(order) + 1)))
ys = np.arange(len(order))[::-1]
for offset, size, color in zip((.16, -.16), sizes, (BLUE, ORANGE)):
    g = within[(within.Split == "test") & (within["Image size"] == size)].set_index("Model").loc[order]
    ax.hlines(ys + offset, g.Low, g.High, color=color, linewidth=2.5, alpha=.65)
    ax.plot(g["AUC in this size"], ys + offset, "o", color=color, markersize=7, markeredgecolor="white", label=size)
    for y, v, h in zip(ys, g["AUC in this size"], g.High):
        ax.text(h + .008, y + offset, f"{v:.2f}", va="center", fontsize=8.5, color=INK)
ax.axvline(.5, color="#AEB6C5", linestyle="--", linewidth=1)
ax.set_yticks(ys, order); ax.set_xlim(.45, 1.03); ax.set_xlabel("Test AUC within one image size"); ax.set_title("Accuracy inside each image size"); ax.grid(axis="y", visible=False); ax.legend(ncols=2, loc="lower left")
fig.tight_layout(); plt.show()

## Check 3: does the score give away the photo type?

We only use **non-referable** photos here, so disease can't explain the result. A value near 0.5 means the score doesn't depend on image size. A value near 1 means the model scores one photo type much higher, even when both are not referable.

In [ ]:
leak = []
for slug, p in predictions.items():
    part = p[(p.split == "test") & (p.referable == 0) & p.image_size.isin(common)]
    leak.append({"Model": metrics[slug]["model"], "Photos": len(part), "Score separates image size (AUC)": roc_auc_score(part.image_size == sizes[-1], part.score)})
leak = pd.DataFrame(leak).set_index("Model").loc[order].reset_index()
print(leak.to_string(index=False, formatters={"Score separates image size (AUC)": "{:.3f}".format}))

## Check 4: does the same cut-off work on both sizes?

The app uses one cut-off for every photo. If sensitivity or specificity differs a lot between sizes, that cut-off may behave differently on a new camera too.

In [ ]:
def rates(y, score, threshold):
    tn, fp, fn, tp = confusion_matrix(y, np.asarray(score) >= threshold, labels=[0, 1]).ravel()
    return tp / max(1, tp + fn), tn / max(1, tn + fp)


cutoff = []
for slug, p in predictions.items():
    for size, g in p[(p.split == "test") & p.image_size.isin(common)].groupby("image_size"):
        sens, spec = rates(g.referable, g.score, metrics[slug]["threshold"])
        cutoff.append({"Model": metrics[slug]["model"], "Image size": size, "Sensitivity": sens, "Specificity": spec})
cutoff = pd.DataFrame(cutoff).set_index("Model").loc[order].reset_index()
print(cutoff.to_string(index=False, formatters={"Sensitivity": "{:.1%}".format, "Specificity": "{:.1%}".format}))

## Save the results for notebook 10

In [ ]:
gaps = within.groupby(["slug", "Model", "Split"]).agg(overall=("Overall AUC", "first"), weakest=("AUC in this size", "min"), mean_within=("AUC in this size", "mean")).reset_index()
gaps["fall"] = gaps.overall - gaps.weakest
gaps.to_csv(RESULTS / "camera_bias.csv", index=False)
print(gaps[gaps.Split == "test"].drop(columns=["slug", "Split"]).round(3).to_string(index=False))
print("\nSaved results/camera_bias.csv")

## What this can and can't show

- **A small check:** this only compares two image sizes from one dataset. A good result doesn't prove the model will work on a new camera.
- **The real test:** notebook 11 scores photos the model has never seen, from another dataset and from your adapter.